# 🏷️ 04c — Reetiquetado con Lead Times Corregidos
## Cambios respecto a 04 original

| Familia | 04 original | 04c | Motivo |
|---------|------------|-----|--------|
| `yaw_cable` | 168h | **72h** | Con 168h el 47% del dataset es positivo — demasiado fácil, modelo no aprende |
| `brake_hydro` | 120h | **120h** | Sin cambio — señal ya física |
| `generator` | 120h | **120h** | Sin cambio |
| `pitch_bat` | 336h | **336h** | Sin cambio |

Genera `dataset_labeled_v2.parquet` — **no sobreescribe el original**.


In [1]:
import os
import pandas as pd
import numpy as np

base_dir = os.path.dirname(os.getcwd())
silver_dir = os.path.join(base_dir, 'data', 'silver')

telem   = pd.read_parquet(os.path.join(silver_dir, 'turbine_1_telemetry_clean.parquet'))
targets = pd.read_parquet(os.path.join(silver_dir, 'fault_targets_grouped.parquet'))
telem   = telem.sort_values('timestamp').reset_index(drop=True)

print(f'Telemetría: {len(telem):,} filas')
print(f'Eventos:    {len(targets):,}')
print(f'Rango:      {telem["timestamp"].min()} → {telem["timestamp"].max()}')


Telemetría: 210,384 filas
Eventos:    330
Rango:      2017-12-31 23:00:00 → 2021-12-31 22:50:00


In [2]:
# ==============================================================================
# CONFIGURACIÓN — lead times corregidos
# ==============================================================================
FAULT_FAMILIES = {
    'yaw_cable':   {'lead_hours': 72},   # 168h → 72h: evita que 47% del dataset sea positivo
    'brake_hydro': {'lead_hours': 120},
    'generator':   {'lead_hours': 120},
    'pitch_bat':   {'lead_hours': 336},
}

def label_family(telem_df, fault_times_sorted, lead_hours):
    ts_array    = telem_df['timestamp'].values.astype('datetime64[ns]')
    fault_array = np.array(fault_times_sorted, dtype='datetime64[ns]')
    hours_arr   = np.full(len(ts_array), np.nan)
    for i, ts in enumerate(ts_array):
        future = fault_array[fault_array > ts]
        if len(future) == 0:
            continue
        delta_h = (future[0] - ts) / np.timedelta64(1, 'h')
        if delta_h <= lead_hours:
            hours_arr[i] = delta_h
    return hours_arr

import time
for family, cfg in FAULT_FAMILIES.items():
    t0 = time.time()
    fault_times = sorted(targets[targets['family'] == family]['timestamp'].tolist())
    hours_arr   = label_family(telem, fault_times, cfg['lead_hours'])
    telem[f'hours_to_{family}'] = hours_arr
    telem[f'is_pre_{family}']   = ~np.isnan(hours_arr)
    n_pos = telem[f'is_pre_{family}'].sum()
    pct   = 100 * n_pos / len(telem)
    print(f'{family:<15}: {n_pos:>7,} positivos ({pct:>5.1f}%)  lead={cfg["lead_hours"]}h  [{time.time()-t0:.0f}s]')

output_path = os.path.join(silver_dir, 'dataset_labeled_v2.parquet')
telem.to_parquet(output_path, index=False)
print(f'\n✅ Guardado: dataset_labeled_v2.parquet')
print(f'   Objetivo yaw_cable: ~10-20% positivos (antes era 47%)')


yaw_cable      :  52,639 positivos ( 25.0%)  lead=72h  [1s]
brake_hydro    :  21,185 positivos ( 10.1%)  lead=120h  [2s]
generator      :  22,272 positivos ( 10.6%)  lead=120h  [1s]
pitch_bat      :  28,753 positivos ( 13.7%)  lead=336h  [1s]

✅ Guardado: dataset_labeled_v2.parquet
   Objetivo yaw_cable: ~10-20% positivos (antes era 47%)
